<a href="https://colab.research.google.com/github/arturfurlan07-debug/MINI_PROJETO-_3/blob/main/mini_projeto_3_api_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Terceiro Mini Projeto – Consumo de APIs
## API de Avaliações da Loja do NARGAS (FastAPI + requests)

Este notebook cria a estrutura de pastas pedida, sobe o servidor dentro do Colab e roda o cliente.
Execute as células **na ordem**.


### 1) Instalar dependências

In [1]:
!pip install -q fastapi uvicorn requests

### 2) Criar a estrutura de pastas

In [2]:
import os

os.makedirs('meu_projeto_api/server/app', exist_ok=True)
os.makedirs('meu_projeto_api/client', exist_ok=True)
%cd meu_projeto_api

/content/meu_projeto_api


### 3) Criar os arquivos do projeto

In [3]:
%%writefile requirements.txt
fastapi
uvicorn
requests


Writing requirements.txt


In [4]:
%%writefile server/app/__init__.py
# pacote do servidor


Writing server/app/__init__.py


In [5]:
%%writefile server/app/sentimento.py
"""Análise de sentimento simples (baseada em léxico) para textos em português.

Não usa bibliotecas externas: assim o servidor sobe rápido e o código é fácil
de explicar. Para evoluir, dá para trocar `analisar_texto` por um modelo de IA
(ex.: transformers) sem mudar o resto da API.
"""
import re
import unicodedata

POSITIVAS = {
    "bom", "boa", "otimo", "otima", "excelente", "maravilhoso", "maravilhosa",
    "adorei", "amei", "gostei", "recomendo", "perfeito", "perfeita", "rapido",
    "rapida", "eficiente", "legal", "incrivel", "satisfeito", "satisfeita",
    "feliz", "agradavel", "barato", "qualidade", "top", "show", "nota10",
    "atencioso", "atenciosa", "confiavel", "facil", "lindo", "linda",
}

NEGATIVAS = {
    "ruim", "pessimo", "pessima", "horrivel", "terrivel", "odiei", "detestei",
    "lento", "lenta", "demorado", "demorada", "caro", "cara", "defeito",
    "quebrado", "quebrada", "decepcionado", "decepcionada", "decepcionante",
    "problema", "problemas", "fraco", "fraca", "falso", "falsa", "golpe",
    "lixo", "insatisfeito", "insatisfeita", "arrependido", "arrependida",
    "dificil", "sujo", "suja", "grosso", "grossa", "enganoso", "enganosa",
}

NEGACOES = {"nao", "nunca", "jamais", "nem", "sem"}


def normalizar(texto: str) -> str:
    """Minúsculas e sem acentos: 'Ótimo' vira 'otimo'."""
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")


def tokenizar(texto: str) -> list:
    return re.findall(r"[a-z0-9]+", normalizar(texto))


def analisar_texto(texto: str) -> dict:
    """Devolve {'sentimento', 'pontuacao', 'confianca', 'palavras_chave'}.

    Uma negação logo antes da palavra inverte o sinal ("não gostei" -> negativo).
    """
    palavras = tokenizar(texto)
    pontuacao = 0
    chaves = []
    for i, palavra in enumerate(palavras):
        if palavra in POSITIVAS:
            valor = 1
        elif palavra in NEGATIVAS:
            valor = -1
        else:
            continue
        if i > 0 and palavras[i - 1] in NEGACOES:
            valor = -valor
        pontuacao += valor
        chaves.append(palavra)

    if pontuacao > 0:
        sentimento = "positivo"
    elif pontuacao < 0:
        sentimento = "negativo"
    else:
        sentimento = "neutro"

    confianca = 0.5 if not chaves else min(1.0, 0.5 + abs(pontuacao) / (2 * len(chaves)))
    if sentimento == "neutro" and chaves:
        confianca = 0.5  # sinais opostos que se cancelam

    return {
        "sentimento": sentimento,
        "pontuacao": pontuacao,
        "confianca": round(confianca, 2),
        "palavras_chave": chaves,
    }


Writing server/app/sentimento.py


In [6]:
%%writefile server/app/main.py
"""Servidor da API de Análise de Sentimento (FastAPI).

Rodar a partir da raiz do projeto:
    uvicorn server.app.main:app --reload
Documentação automática: http://127.0.0.1:8000/docs
"""
import os
from collections import Counter
from datetime import datetime
from typing import Optional

from fastapi import Depends, FastAPI, Header, HTTPException, Query
from pydantic import BaseModel, Field

from .sentimento import analisar_texto

LOJA = os.getenv("LOJA_NOME", "Loja do Nargas")

app = FastAPI(
    title=f"Avaliações – {LOJA}",
    description="Recebe avaliações de clientes da loja e classifica cada uma em positivo, neutro ou negativo.",
    version="1.0.0",
)

# Se API_KEY estiver definida no ambiente (.env), as rotas exigem o header X-API-Key.
API_KEY = os.getenv("API_KEY", "")

# "Banco de dados" em memória (some quando o servidor reinicia).
historico = []
proximo_id = 1


def verificar_chave(x_api_key: Optional[str] = Header(default=None)):
    if API_KEY and x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="Chave de API inválida ou ausente.")


class TextoEntrada(BaseModel):
    texto: str = Field(..., min_length=3, max_length=1000, examples=["Adorei o atendimento, recomendo!"])
    autor: str = Field(default="Anônimo", max_length=50, examples=["Maria"])


class Resultado(BaseModel):
    id: int
    autor: str
    texto: str
    sentimento: str
    pontuacao: int
    confianca: float
    palavras_chave: list
    data_hora: str


@app.get("/")
def raiz():
    return {"mensagem": f"API de avaliações da {LOJA} no ar.", "docs": "/docs"}


@app.post("/analisar", response_model=Resultado, status_code=201, dependencies=[Depends(verificar_chave)])
def analisar(entrada: TextoEntrada):
    global proximo_id
    resultado = analisar_texto(entrada.texto)
    registro = {
        "id": proximo_id,
        "autor": entrada.autor,
        "texto": entrada.texto,
        "data_hora": datetime.now().isoformat(timespec="seconds"),
        **resultado,
    }
    proximo_id += 1
    historico.append(registro)
    return registro


@app.get("/historico", response_model=list[Resultado], dependencies=[Depends(verificar_chave)])
def listar_historico(
    sentimento: Optional[str] = Query(default=None, description="positivo, negativo ou neutro"),
    limite: int = Query(default=20, ge=1, le=100),
):
    itens = historico
    if sentimento:
        itens = [r for r in itens if r["sentimento"] == sentimento.lower()]
    return itens[-limite:]


@app.get("/historico/{id_analise}", response_model=Resultado, dependencies=[Depends(verificar_chave)])
def buscar_analise(id_analise: int):
    for registro in historico:
        if registro["id"] == id_analise:
            return registro
    raise HTTPException(status_code=404, detail="Análise não encontrada.")


@app.get("/estatisticas", dependencies=[Depends(verificar_chave)])
def estatisticas():
    total = len(historico)
    contagem = Counter(r["sentimento"] for r in historico)
    def pct(chave):
        return round(100 * contagem.get(chave, 0) / total, 1) if total else 0.0

    return {
        "loja": LOJA,
        "total": total,
        "positivo": contagem.get("positivo", 0),
        "neutro": contagem.get("neutro", 0),
        "negativo": contagem.get("negativo", 0),
        "percentual_positivo": pct("positivo"),
        "percentual_neutro": pct("neutro"),
        "percentual_negativo": pct("negativo"),
    }


@app.delete("/historico", dependencies=[Depends(verificar_chave)])
def limpar_historico():
    removidos = len(historico)
    historico.clear()
    return {"removidos": removidos}


Writing server/app/main.py


In [7]:
%%writefile client/main.py
"""Cliente que consome a API de avaliações da loja (análise de sentimento).

Rodar (com o servidor no ar):
    python client/main.py
Variáveis de ambiente (opcionais, veja .env.example):
    API_URL  endereço do servidor (padrão: http://127.0.0.1:8000)
    API_KEY  chave enviada no header X-API-Key (só se o servidor exigir)
"""
import os

import requests

API_URL = os.getenv("API_URL", "http://127.0.0.1:8000").rstrip("/")
API_KEY = os.getenv("API_KEY", "")
HEADERS = {"X-API-Key": API_KEY} if API_KEY else {}
TIMEOUT = 10


def chamar(metodo, caminho, **kwargs):
    """Faz a requisição e trata os erros mais comuns."""
    try:
        resposta = requests.request(metodo, f"{API_URL}{caminho}", headers=HEADERS,
                                    timeout=TIMEOUT, **kwargs)
        resposta.raise_for_status()
        return resposta.json()
    except requests.exceptions.ConnectionError:
        print(f"Não consegui conectar em {API_URL}. O servidor está no ar?")
    except requests.exceptions.HTTPError as erro:
        print(f"Erro da API ({erro.response.status_code}): {erro.response.text}")
    except requests.exceptions.Timeout:
        print("A API demorou demais para responder.")
    return None


def analisar(texto, autor="Anônimo"):
    return chamar("POST", "/analisar", json={"texto": texto, "autor": autor})


def historico(sentimento=None, limite=20):
    params = {"limite": limite}
    if sentimento:
        params["sentimento"] = sentimento
    return chamar("GET", "/historico", params=params)


def estatisticas():
    return chamar("GET", "/estatisticas")


def limpar():
    return chamar("DELETE", "/historico")


def mostrar_resultado(r):
    print(f"#{r['id']} {r['autor']} - [{r['sentimento'].upper()}] confiança {r['confianca']:.0%} "
          f"| palavras: {', '.join(r['palavras_chave']) or '-'}")
    print(f"     \"{r['texto']}\"")


def mostrar_painel():
    """Painel de avaliações da loja: quantidade e barra de cada sentimento."""
    est = estatisticas()
    if not est:
        return
    print(f"\n===== AVALIAÇÕES - {est['loja']} ({est['total']} no total) =====")
    for nome, chave in (("Positivas", "positivo"), ("Neutras", "neutro"), ("Negativas", "negativo")):
        pct = est[f"percentual_{chave}"]
        barra = "█" * round(pct / 5)
        print(f"{nome:<10} {barra:<20} {pct:>5}% ({est[chave]})")


def demo():
    """Envia avaliações de exemplo de clientes e mostra o painel da loja."""
    avaliacoes = [
        ("Maria", "Adorei o atendimento, produto de ótima qualidade. Recomendo!"),
        ("João", "Péssimo, chegou quebrado e a entrega foi muito demorada."),
        ("Ana", "Não gostei da embalagem, mas o produto é bom."),
        ("Pedro", "Chegou na terça-feira."),
        ("Carla", "Entrega rápida e preço barato, super satisfeito!"),
        ("Lucas", "Horrível, não recomendo. Tive problemas com o suporte."),
    ]
    print("== Enviando avaliações dos clientes ==")
    for autor, texto in avaliacoes:
        resultado = analisar(texto, autor)
        if resultado is None:
            return
        mostrar_resultado(resultado)

    mostrar_painel()

    print("\n== Avaliações negativas ==")
    for r in historico(sentimento="negativo") or []:
        mostrar_resultado(r)


MENU = """
===== AVALIAÇÕES DA LOJA =====
1 Enviar uma avaliação
2 Ver avaliações
3 Ver painel (positivas/neutras/negativas)
4 Limpar avaliações
5 Rodar demonstração
0 Sair"""


def menu():
    while True:
        print(MENU)
        op = input("Opção: ").strip()
        if op == "0":
            break
        elif op == "1":
            autor = input("Seu nome (Enter = anônimo): ").strip() or "Anônimo"
            r = analisar(input("Sua avaliação: ").strip(), autor)
            if r:
                mostrar_resultado(r)
        elif op == "2":
            filtro = input("Filtrar (positivo/negativo/neutro, Enter = todos): ").strip() or None
            for r in historico(filtro) or []:
                mostrar_resultado(r)
        elif op == "3":
            mostrar_painel()
        elif op == "4":
            print(limpar())
        elif op == "5":
            demo()
        else:
            print("Opção inválida.")


if __name__ == "__main__":
    menu()


Writing client/main.py


### 4) Subir o servidor em segundo plano
O Colab roda o servidor como um processo separado e o cliente chama `http://127.0.0.1:8000`.

In [8]:
import subprocess, time, requests

servidor = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server.app.main:app', '--port', '8000'],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)

for _ in range(20):  # espera o servidor ficar pronto
    try:
        print(requests.get('http://127.0.0.1:8000/').json())
        break
    except requests.exceptions.ConnectionError:
        time.sleep(0.5)
else:
    print('O servidor não subiu. Rode: !python -m uvicorn server.app.main:app --port 8000')

{'mensagem': 'API de avaliações da Loja do Nargas no ar.', 'docs': '/docs'}


### 5) Rodar o cliente (demonstração)

In [9]:
import sys

sys.path.insert(0, 'client')
import main as cliente

cliente.limpar()
cliente.demo()

== Enviando avaliações dos clientes ==
#1 Maria - [POSITIVO] confiança 100% | palavras: adorei, otima, qualidade, recomendo
     "Adorei o atendimento, produto de ótima qualidade. Recomendo!"
#2 João - [NEGATIVO] confiança 100% | palavras: pessimo, quebrado, demorada
     "Péssimo, chegou quebrado e a entrega foi muito demorada."
#3 Ana - [NEUTRO] confiança 50% | palavras: gostei, bom
     "Não gostei da embalagem, mas o produto é bom."
#4 Pedro - [NEUTRO] confiança 50% | palavras: -
     "Chegou na terça-feira."
#5 Carla - [POSITIVO] confiança 100% | palavras: rapida, barato, satisfeito
     "Entrega rápida e preço barato, super satisfeito!"
#6 Lucas - [NEGATIVO] confiança 100% | palavras: horrivel, recomendo, problemas
     "Horrível, não recomendo. Tive problemas com o suporte."

===== AVALIAÇÕES - Loja do Nargas (6 no total) =====
Positivas  ███████               33.3% (2)
Neutras    ███████               33.3% (2)
Negativas  ███████               33.3% (2)

== Avaliações negativas

### 6) Enviar uma avaliação e ver o painel
Troque o nome e o texto e rode de novo.

In [10]:
r = cliente.analisar('Atendimento ótimo, mas o preço está caro.', 'Seu nome')
cliente.mostrar_resultado(r)
cliente.mostrar_painel()

#7 Seu nome - [NEUTRO] confiança 50% | palavras: otimo, caro
     "Atendimento ótimo, mas o preço está caro."

===== AVALIAÇÕES - Loja do Nargas (7 no total) =====
Positivas  ██████                28.6% (2)
Neutras    █████████             42.9% (3)
Negativas  ██████                28.6% (2)


### 7) (Opcional) Menu interativo do cliente
Abre o menu com `input()`; digite `0` para sair.

In [ ]:
cliente.menu()


===== AVALIAÇÕES DA LOJA =====
1 Enviar uma avaliação
2 Ver avaliações
3 Ver painel (positivas/neutras/negativas)
4 Limpar avaliações
5 Rodar demonstração
0 Sair
Opção: 1
Seu nome (Enter = anônimo): Artur
Sua avaliação: postivo
Não consegui conectar em http://127.0.0.1:8000. O servidor está no ar?

===== AVALIAÇÕES DA LOJA =====
1 Enviar uma avaliação
2 Ver avaliações
3 Ver painel (positivas/neutras/negativas)
4 Limpar avaliações
5 Rodar demonstração
0 Sair


### 8) Encerrar o servidor

In [12]:
servidor.terminate()